# Import libraries

In [ ]:
import os
import logging
from langchain_chroma import Chroma
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_classic.retrievers import MultiQueryRetriever
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

# Connect to ChromaDB

In [2]:
api_key = os.getenv("API_KEY")
llm = model = GoogleGenerativeAI(
    api_key=api_key,
    model="gemini-2.5-flash-lite",
    temperature=0.0,
    max_tokens=50000,
    timeout=None,
    max_retries=2
)

collection_name = "langchain_docs_index"
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001", api_key=api_key)

vectorstore = Chroma(embedding_function=embedding, collection_name=collection_name, persist_directory="./data/vectors/chroma_db")

# Initialize retriever

In [3]:
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm,
)

In [4]:
retriever.invoke(
    "What is BPE?"
)

[Document(id='a72c54f8-9126-5dc7-b01d-b4343aa013ac', metadata={'contains_math_latex': False, 'importance_score': 'High', 'talks_about_language_modeling': False, 'linguistic_focus': '', 'contains_code_or_cli': False, 'contains_regex': False, 'source': 'data\\raw\\book_chapter_02.pdf', 'creationdate': 'D:20260329110007', 'volume': 1, 'total_pages': 34, 'talks_about_morphology': False, 'content_type': 'Examples', 'talks_about_ngrams': False, 'chapter': 2, 'contains_table': False, 'page': 11, 'talks_about_tokenization': True}, page_content='re-:\ncorpus\nvocabulary\n2\nnew\n, e, n, r, s, t, w, ne, new,\nr,\nre\n2\nre new\n1\ns e t\n1\nre s e t\nIf we continue, the next merges are:\nmerge\ncurrent vocabulary\n( , new)\n, e, n, r, s, t, w, ne, new,\nr,\nre,\nnew\n( re, new)\n, e, n, r, s, t, w, ne, new,\nr,\nre,\nnew,\nrenew\n(s, e)\n, e, n, r, s, t, w, ne, new,\nr,\nre,\nnew,\nrenew, se\n(se, t)\n, e, n, r, s, t, w, ne, new,\nr,\nre,\nnew,\nrenew, se, set\nfunction BYTE-PAIR ENCODING(string

# Improve query retriever with custom question generation chain

In [5]:
# Definición de esquema de salida de preguntas
class LineListOutputParser(BaseOutputParser):
    def parse(self, text: str):
        # Splits the output by newline and removes empty lines
        return [line.strip() for line in text.strip().split("\n") if line.strip()]

In [6]:
# Creación de prompt personalizado
prompt = PromptTemplate.from_template(
    """You are an AI language assistant well versed in LLMs.
Your more precise task is to generate five different versions of the given question to retrieve relevant documents from a vector database.
By generating multiple perspectives on the question, your goal is to overcome some of the limitations of the distance-based similarity search.

Provide these alternative questions separed by newlines.

Original question: {question}
New questions:"""
)

# In language expression language, you could create the chain with:
llm_chain = prompt | llm | LineListOutputParser()
llm_chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='You are an AI language assistant well versed in LLMs.\nYour more precise task is to generate five different versions of the given question to retrieve relevant documents from a vector database.\nBy generating multiple perspectives on the question, your goal is to overcome some of the limitations of the distance-based similarity search.\n\nProvide these alternative questions separed by newlines.\n\nOriginal question: {question}\nNew questions:')
| GoogleGenerativeAI(google_api_key=SecretStr('**********'), model='models/gemini-2.5-flash-lite', temperature=0.0, max_output_tokens=50000, max_retries=2, client=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reas

In [7]:
# Uso de cadena de generación de preguntas personalizada
llm_chain.invoke(
    {"question": "What is BPE?"}
)

['What is Byte Pair Encoding and how does it work?',
 'Explain the concept of Byte Pair Encoding (BPE) in natural language processing.',
 'Describe the algorithm behind Byte Pair Encoding.',
 'What are the applications and benefits of using Byte Pair Encoding?',
 'How is Byte Pair Encoding used for subword tokenization in LLMs?']

In [8]:
# Integración de cadena de generación de preguntas personalizada en retriever
retriever = MultiQueryRetriever(
    retriever=vectorstore.as_retriever(),
    llm_chain=llm_chain
)

In [9]:
# Uso de retriever con cadena de generación de preguntas personalizada
retriever.invoke(
    "What are Morphemes?"
)

[Document(id='1e6dccce-89c9-5043-a6f9-b8c9c80c5967', metadata={'talks_about_language_modeling': False, 'talks_about_morphology': True, 'creationdate': 'D:20260329110007', 'chapter': 2, 'volume': 1, 'importance_score': 'High', 'content_type': 'Narrative', 'contains_code_or_cli': False, 'total_pages': 34, 'source': 'data\\raw\\book_chapter_02.pdf', 'contains_regex': False, 'talks_about_tokenization': False, 'linguistic_focus': 'English, Chinese', 'page': 4, 'contains_math_latex': False, 'talks_about_ngrams': False, 'contains_table': False}, page_content='morpheme\nword fox consists of one morpheme (the morpheme fox) while the word cats consists\nof two: the morpheme cat and the morpheme -s that indicates plural.\nHere’s a sentence in English segmented into morphemes with hyphens:\n(2.6) Doc work-ed care-ful-ly wash-ing the glass-es\nAs we mentioned above, in Chinese, conveniently, the writing system is set up\nso that each character mainly describes a morpheme. Here’s a sentence in Manda